# Primera práctica Machine Learning
Manuel Manchado Barquero, NIA:100522228

David Mateu Serrano, NIA:100522337

[Github](https://github.com/100522228/P1_Machine_learning.git)

## EDA

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

SEED = 100522228

# Importamos el dataframe
file_path = Path("content/P1") / "bank_10.pkl"
df = pd.read_pickle(file_path)

#print(df.head(10))

# Vemos el número de variables e instancias
n_instancias, n_variables = df.shape
print(f"{n_instancias} instancias y {n_variables} variables.")

# Agrupamos los tipos de variables (a priori no podemos saber cuales son categoricas u ordinales)
lista_variables = list(df.columns)
print(lista_variables)
var_categoricas_y_ordinales = []
var_numericas = []
for var in lista_variables:
  if df[var].dtype == 'int64':
    var_numericas.append(var)
  elif var == "deposit":
    continue
  else:
    var_categoricas_y_ordinales.append(var)

print(f'{var_numericas} variables numéricas.')
print(f'{var_categoricas_y_ordinales} variables categoricas y ordinales.') # Después de imprimir el df entero, vemos que education es ordinal

# Vemos cardinalidad variables categóricas sin contar NaNs
var_categoricas = [v for v in var_categoricas_y_ordinales if v != "education"]
var_ordinal = ['education']
for var in var_categoricas:
    print(var, df[var].nunique()) # Vemos que tanto job como month tienen alta cardinalidad (>10 valores únicos)



11000 instancias y 17 variables.
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'deposit']
['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous'] variables numéricas.
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome'] variables categoricas y ordinales.
job 12
marital 3
default 2
housing 2
loan 2
contact 3
month 12
poutcome 4


## Missing values

In [7]:
# Vamos a mirar los valores de las variables categóricas buscando posibles celdas con valores
# missing distintos de None o Null en este caso encontramos 'unknown'
for col in lista_variables:
    print(col, df[col].unique())

# Nótese que job tiene los 2 valores None y 'unknown' 

print("\n---\n")

# Vamos a ver cuantos valores hay missing o unknown de cada feature y que porcentaje representan
n = len(df)
for col in lista_variables:
    n_nan = df[col].isna().sum()
    n_unknown = (df[col] == 'unknown').sum()
    
    if n_nan > 0:
        print(col, "NaN:", n_nan, f"({n_nan/n:.2%})")   
    if n_unknown > 0:
        print(col, "unknown:", n_unknown, f"({n_unknown/n:.2%})")

#Pdays al ser un caso concreto lo veremos ahora
n_minus = (df['pdays']==-1).sum()
print("pdays -1: ", n_minus, f"({n_minus/n:0.2%})")

#hacemos ya el preprocesado de pdays
#TODO: Explicar un poco

# Crear variable binaria: 1 si hubo contacto previo, 0 si no
df['contact'] = (df['pdays'] != -1).astype(int)

# Reemplazar los -1 por 0 en pdays
df['pdays'] = df['pdays'].replace(-1, 0)

age [59 56 41 55 54 42 60 37 28 38 30 29 46 31 35 32 49 43 26 40 33 23 48 45
 36 52 53 39 57 51 44 24 50 27 34 47 25 58 61 68 75 22 69 66 85 72 90 67
 71 21 74 65 62 83 70 76 77 19 73 63 20 78 95 64 79 82 18 86 84 87 92 81
 80 93 88 89]
job ['admin.' None 'technician' 'management' 'retired' 'services'
 'blue-collar' 'unemployed' 'entrepreneur' 'housemaid' 'unknown'
 'self-employed' 'student']
marital ['married' None 'single' 'divorced']
education ['secondary' 'tertiary' 'primary' 'unknown']
default ['no' 'yes']
balance [2343   45 1270 ... 1594 2593 -134]
housing ['yes' 'no']
loan ['no' 'yes']
contact ['unknown' 'cellular' 'telephone']
day [ 5  6  7  8  9 12 13 14 15 16 19 20 21 23 26 27 28 29 30  2  3  4 11 17
 18 24  1 10 22 25 31]
month ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'jan' 'feb' 'mar' 'apr' 'sep']
duration [1042 1467 1389 ... 1504  818 1812]
campaign [ 1  2  3  4  6  5  8 11  9 10 15 12 14  7 13 17 29 21 20 16 32 19 25 24
 22 43 18 41 63 27 30 26 23 28 33 31]
pdays [ -1 1

Podemos ver que hay bastantes features que toman valor 'unknow' parece razonable pensar este valor como missing value.

In [8]:
# Columnas constantes

for col in lista_variables:
    # Obtenemos las frecuencias de todos los valores (incluyendo NaNs)
    counts = df[col].value_counts(dropna=False, normalize=True)
    top_freq = counts.iloc[0]
    nunique = len(counts)

    print(f"{col:<12} | Únicos: {nunique:<5} | Frecuencia mayoritaria: {top_freq:>7.2%} | Valor mayoritario: {counts.index[0]:<10}")

age          | Únicos: 76    | Frecuencia mayoritaria:   4.46% | Valor mayoritario: 31        
job          | Únicos: 13    | Frecuencia mayoritaria:  22.28% | Valor mayoritario: management
marital      | Únicos: 4     | Frecuencia mayoritaria:  56.04% | Valor mayoritario: married   
education    | Únicos: 4     | Frecuencia mayoritaria:  49.03% | Valor mayoritario: secondary 
default      | Únicos: 2     | Frecuencia mayoritaria:  98.48% | Valor mayoritario: no        
balance      | Únicos: 3783  | Frecuencia mayoritaria:   6.88% | Valor mayoritario: 0         
housing      | Únicos: 2     | Frecuencia mayoritaria:  52.74% | Valor mayoritario: no        
loan         | Únicos: 2     | Frecuencia mayoritaria:  86.99% | Valor mayoritario: no        
contact      | Únicos: 2     | Frecuencia mayoritaria:  74.57% | Valor mayoritario: 0         
day          | Únicos: 31    | Frecuencia mayoritaria:   5.11% | Valor mayoritario: 20        
month        | Únicos: 12    | Frecuencia mayorita

Vemos que la feature default es casi constante con un 98.48% de valores 'no'

También podemos ver que no hay columnas ID.

Este problema se trata de un problema de clasificación por lo que es muy importante ver si está desbalanceado.
En la salida anterior se puede ver que deposit no está desbalanceada poque tiene un 52.55% de valores 'no' y un 47.45% de valores 'si'.

## Definición evaluación

In [ ]:
from sklearn.model_selection import train_test_split


# Caso 1: None -> 'unknown'
df_unknown = df.copy()
df_unknown['job'] = df_unknown['job'].fillna('unknown')

X_unknown = df_unknown.drop(columns=['deposit'])
y_unknown = df_unknown['deposit']

X_train_u, X_test_u, y_train_u, y_test_u = train_test_split(
    X_unknown,
    y_unknown,
    test_size=0.33,
    random_state=SEED
)

# Caso 2: None -> 'nada'
df_nada = df.copy()

# None -> 'nada'
df_nada['job'] = df_nada['job'].fillna('nada')

X_nada = df_nada.drop(columns=['deposit'])
y_nada = df_nada['deposit']

X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_nada,
    y_nada,
    test_size=0.33,
    random_state=SEED
)


Descripción de q vamos a hacer Holdout con 1/3-2/3 y vamos a usar los algoritmos q sea para inner evaluation y el q sea para outer. y q vamos a hacer crossvalidation en inner con K-fold=3 (es lo q pone en el esquema) en principio con accuracy pq la variable desposit no está balanceada y la q tenga mejor accuracy será mejor

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer

# Definimos la validación cruzada de 3 folds
cv_inner = KFold(n_splits=3, shuffle=True, random_state=SEED)

scalers = {
    'Standard': StandardScaler(),
    'MinMax': MinMaxScaler(),
    'Robust': RobustScaler(),
    'No Scaler': 'passthrough'
}

results_unknown = {}
results_nada = {}


for name, scaler in scalers.items():

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', scaler, var_numericas),
            ('cat', OneHotEncoder(handle_unknown='ignore'), var_categoricas_y_ordinales),
        ],
        remainder='passthrough' # Ignora la feature contacts pq 'hacemos el one hot encoder a mano'
    )

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', KNeighborsClassifier())
    ])

    scores = cross_val_score(
        pipeline,
        X_train_u,
        y_train_u,
        cv=cv_inner,
        scoring='accuracy'
    )

    results_unknown[name] = {
        'mean': scores.mean(),
        'std': scores.std()
    }

    print(f"{name:<12} | Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

best_scaler_unknown = max(results_unknown, key=lambda x: results_unknown[x]['mean'])
print(f"\nEl mejor escalador para UNKNOWN es: {best_scaler_unknown}\n")


for name, scaler in scalers.items():

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', scaler, var_numericas),
            ('cat', OneHotEncoder(handle_unknown='ignore'), var_categoricas_y_ordinales),
        ],
        remainder='passthrough' # Ignora la feature contacts
    )

    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', KNeighborsClassifier())
    ])

    scores = cross_val_score(
        pipeline,
        X_train_n,
        y_train_n,
        cv=cv_inner,
        scoring='accuracy'
    )

    results_nada[name] = {
        'mean': scores.mean(),
        'std': scores.std()
    }

    print(f"{name:<12} | Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

best_scaler_nada = max(results_nada, key=lambda x: results_nada[x]['mean'])
print(f"\nEl mejor escalador para NADA es: {best_scaler_nada}")


Standard     | Accuracy media: 0.7849 (+/- 0.0071)
MinMax       | Accuracy media: 0.7156 (+/- 0.0072)
Robust       | Accuracy media: 0.7760 (+/- 0.0116)
No Scaler    | Accuracy media: 0.7449 (+/- 0.0106)

El mejor escalador para UNKNOWN es: Standard

Standard     | Accuracy media: 0.7848 (+/- 0.0071)
MinMax       | Accuracy media: 0.7164 (+/- 0.0076)
Robust       | Accuracy media: 0.7759 (+/- 0.0119)
No Scaler    | Accuracy media: 0.7449 (+/- 0.0106)

El mejor escalador para NADA es: Standard
